# M5. RAG 체인 (05_m5_RAG체인)

질문 → 관련 사고사례 검색 → 프롬프트 구성 → LLM 답변 → 출처 표시까지 연결합니다.

| 체크리스트 | 이 노트북의 위치 |
|---|---|
| M5-1 사용자 질문 입력 | `ask(question)` 함수의 입력 (4절) |
| M5-2 관련 문서 검색 | `retrieve()` (2절) |
| M5-3 검색 결과를 프롬프트에 전달 | `build_prompt()` (3절) |
| M5-4 LLM 답변 생성 | `ask()` 안의 LLM 호출 (4절) |
| M5-5 출처 표시와 근거 부족 처리 | `MIN_SIM` 기준 + 출처 목록 (2·4절) |
| M5-6 질문부터 답변까지 전체 테스트 | 5절 |

**사전 준비**: 04 노트북으로 `data/chroma`에 6,032건이 적재되어 있어야 하고, `.env`에 `OPENAI_API_KEY`가 있어야 합니다.

## 필요 패키지
이 노트북은 **chromadb, openai, python-dotenv, pandas** 를 사용합니다. 프로젝트 환경(uv)에 설치되어 있어야 합니다.
`ModuleNotFoundError`가 나면 터미널에서 아래처럼 설치한 뒤 커널을 재시작하세요.
```
uv add chromadb openai python-dotenv pandas
```

## 0. 준비

In [ ]:
import os
from pathlib import Path
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

# data/chroma가 있는 폴더를 위로 올라가며 찾는다
ROOT = Path.cwd()
while not (ROOT / "data" / "chroma").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "chroma").exists(), "data/chroma가 없습니다. 04 노트북을 먼저 실행하세요."

load_dotenv(ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), f"{ROOT / '.env'} 에 OPENAI_API_KEY=키값 을 넣고 커널을 재시작하세요."
client = OpenAI(timeout=60, max_retries=2)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")
K = 3             # 검색해 올 사례 수
MIN_SIM = 0.40    # 이 값보다 비슷하지 않은 사례는 근거로 쓰지 않는다 (5-1절 점검: 범위 밖 최고 0.353 < 0.40 < 범위 안 최저 0.438)
NO_EVIDENCE = "제공된 사례에서 근거를 찾을 수 없습니다."

col = chromadb.PersistentClient(path=str(ROOT / "data" / "chroma")).get_collection("sif")
assert col.count() > 0, "sif 컬렉션이 비어 있습니다. 04 노트북을 먼저 실행하세요."
print("프로젝트 폴더:", ROOT)
print("컬렉션 문서 수:", col.count(), "| 답변 모델:", CHAT_MODEL)

## 1. 설계
1. 질문을 임베딩해서 가장 비슷한 사례 K건을 찾는다 (필터로 업종 등을 지정할 수 있다).
2. 유사도가 `MIN_SIM`보다 낮은 사례는 버린다. **남은 사례가 없으면 LLM을 부르지 않고** "근거를 찾을 수 없습니다"로 답한다.
3. 남은 사례를 `[사례1]`, `[사례2]`… 형식으로 프롬프트에 넣는다.
4. LLM은 사례 안의 내용만으로 `1) 사고 원인 2) 예방 대책`을 답하고, 근거가 되는 `[사례N]`을 표시한다.
5. 답변 아래에 출처(업종·기인물·재해연도·개요)를 붙인다.

## 2. 관련 문서 검색 (M5-2, M5-5)

In [ ]:
def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=[text[:6000]]).data[0].embedding

def retrieve(question, k=K, where=None, min_sim=None):
    """질문과 비슷한 사례를 유사도 순으로 돌려준다. min_sim 미만은 제외."""
    min_sim = MIN_SIM if min_sim is None else min_sim
    res = col.query(query_embeddings=[embed(question)], n_results=k, where=where)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        sim = 1 - dist
        if sim >= min_sim:
            hits.append({"similarity": round(sim, 3), "document": doc, "meta": meta})
    return hits

# 동작 확인
for h in retrieve("지게차 작업 중 협착 사고"):
    print(h["similarity"], h["meta"]["업종구분"], h["meta"]["기인물"], "|", h["document"].split("[재해개요]")[-1][:50].replace("\n", " "))

## 3. 프롬프트 구성 (M5-3)

In [ ]:
SYSTEM = """당신은 산업재해 예방을 돕는 안전 어시스턴트입니다. 아래 [사례]에 적힌 내용만 근거로 답하세요.
- 답변은 '1) 사고 원인', '2) 예방 대책' 순서로 쓰고, 각 항목마다 근거가 된 사례 번호를 [사례N] 형식으로 표시하세요.
- 사례에 없는 내용은 추측하거나 지어내지 마세요. 법령 조항 번호도 사례에 없으면 쓰지 마세요.
- 사례만으로 질문에 답할 수 없으면 다른 말 없이 정확히 '제공된 사례에서 근거를 찾을 수 없습니다.'라고만 답하세요."""

def build_prompt(question, hits):
    blocks = []
    for n, h in enumerate(hits, 1):
        m = h["meta"]
        head = f"[사례{n}] (업종구분: {m['업종구분']}, 기인물: {m['기인물'] or '미분류'}, 재해연도: {m['재해연도']})"
        blocks.append(head + "\n" + h["document"])
    context = "\n\n".join(blocks)
    return f"{context}\n\n질문: {question}"

demo = retrieve("비계에서 추락한 사고 대책", where={"업종구분": "건설업"})
print(build_prompt("비계에서 추락한 사고 대책", demo)[:800])

## 4. 답변 생성과 출처 표시 (M5-1, M5-4, M5-5)
`ask(질문)`이 전체를 연결합니다. 결과는 `answer`(답변), `sources`(출처), `grounded`(사례에 근거해 답했는지)를 담은 딕셔너리입니다.

In [ ]:
def ask(question, k=K, where=None):
    hits = retrieve(question, k=k, where=where)
    sources = [{
        "사례": n,
        "유사도": h["similarity"],
        "업종구분": h["meta"]["업종구분"],
        "중분류": h["meta"]["중분류"],
        "기인물": h["meta"]["기인물"] or "미분류",
        "재해연도": h["meta"]["재해연도"],
        "개요": h["document"].split("[재해개요]")[-1].split("\n[기인물]")[0].strip()[:80],
    } for n, h in enumerate(hits, 1)]

    if not hits:                       # 근거가 될 사례가 없으면 LLM을 부르지 않는다
        return {"answer": NO_EVIDENCE, "sources": [], "grounded": False, "retrieved": []}

    resp = client.chat.completions.create(
        model=CHAT_MODEL, temperature=0,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": build_prompt(question, hits)}],
    )
    answer = resp.choices[0].message.content.strip()
    grounded = NO_EVIDENCE not in answer
    return {"answer": answer, "sources": sources if grounded else [], "grounded": grounded, "retrieved": sources}

def show(question, **kw):
    r = ask(question, **kw)
    print("Q.", question)
    print(r["answer"])
    if r["sources"]:
        print("\n출처")
        for s in r["sources"]:
            print(f"  [사례{s['사례']}] 유사도 {s['유사도']} | {s['업종구분']} > {s['중분류']} | 기인물 {s['기인물']} | {s['재해연도']}년 | {s['개요']}")
    if not r["grounded"] and r["retrieved"]:
        names = ", ".join(x["기인물"] for x in r["retrieved"])
        print(f"(검색은 됐지만 질문과 맞지 않아 답하지 않음 | 검색된 기인물: {names})")
    print("근거 있음:", r["grounded"])
    print("=" * 80)
    return r

_ = show("지게차 작업 중 협착 사고는 왜 발생하고 어떻게 예방하나요?")

## 5. 전체 테스트 (M5-6)
- **범위 안 질문**: 사례집에 있을 법한 질문 → 근거와 출처가 붙어야 합니다.
- **범위 밖 질문**: 사고 사례와 무관한 질문 → "근거를 찾을 수 없습니다"가 나와야 합니다.
- `where`로 업종 필터를 건 질문도 확인합니다.

In [ ]:
in_scope = [
    "컨베이어에 끼인 사고는 어떻게 예방하나요?",
    "천장크레인으로 중량물을 옮기다 발생한 사고의 원인과 대책은?",
    "이동식 사다리에서 떨어진 사고를 막으려면?",
]
for q in in_scope:
    show(q)

# 업종 필터(건설업)를 건 질문
show("비계에서 추락한 사고를 예방하려면?", where={"업종구분": "건설업"})

In [ ]:
out_of_scope = [
    "오늘 서울 날씨 알려줘",
    "달 탐사선은 언제 발사되나요?",
    "맛있는 파스타 레시피 알려줘",
]
for q in out_of_scope:
    show(q)

### 5-0. 알려진 한계: 구체 용어가 일반 표현에 묻히는 경우
질문에 구체 용어(예: `지붕 채광판`)와 일반 표현(예: `작업하다 추락한 사고`)이 섞여 있으면, 임베딩 검색이 일반 표현 쪽 사례(거푸집·작업발판 등)를 가져올 수 있습니다.
그러면 LLM은 사례가 질문과 맞지 않다고 보고 "근거를 찾을 수 없습니다"로 답합니다. 틀린 답을 지어내지 않는 대신 답을 못 하는 경우입니다.
→ 개선 후보(M7): 질문에서 기인물을 뽑아 메타데이터 필터로 쓰기 / 키워드 검색과 결합하기. 아래에서 같은 주제를 두 가지 표현으로 비교합니다.

In [ ]:
for q in ["지붕 채광판 위에서 작업하다 추락한 사고 대책", "지붕 채광판 파손 추락"]:
    show(q, where={"업종구분": "건설업"})

### 5-1. 유사도 점검 (MIN_SIM 조정용)

In [ ]:
import pandas as pd

def top_sim(q):
    res = col.query(query_embeddings=[embed(q)], n_results=1)
    return round(1 - res["distances"][0][0], 3)

rows = [{"구분": "범위 안", "질문": q, "최고 유사도": top_sim(q)} for q in in_scope] + \
       [{"구분": "범위 밖", "질문": q, "최고 유사도": top_sim(q)} for q in out_of_scope]
chk = pd.DataFrame(rows)
display(chk)
lo_in = chk[chk["구분"] == "범위 안"]["최고 유사도"].min()
hi_out = chk[chk["구분"] == "범위 밖"]["최고 유사도"].max()
print(f"범위 안 최저 {lo_in} / 범위 밖 최고 {hi_out} / 현재 MIN_SIM = {MIN_SIM}")
print("→ MIN_SIM은 범위 밖 최고값보다 크고 범위 안 최저값보다 작아야 합니다.")

## 정리
- 질문 → 검색(K=3, `MIN_SIM` 기준) → 사례 기반 프롬프트 → 답변 + 출처까지 `ask()` 한 번으로 동작합니다.
- 근거가 없으면 LLM을 호출하지 않고 "제공된 사례에서 근거를 찾을 수 없습니다."로 답합니다.
- 다음 단계(M6): 평가 질문 세트를 만들어 검색 품질(Hit@K, MRR)을 측정합니다.